In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Define model ID
model_id = "tiiuae/Falcon-H1-3B-Instruct"

# 2. Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,  # Recommended precision for Falcon-H1
    device_map="auto"           # Automatically dispatches to GPU/CPU
)

# 3. Format input prompt
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain the concept of quantum computing in simple terms."}
]

# Apply chat template for Instruct models
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# 4. Tokenize inputs
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# 5. Generate output
# Tip: Falcon-H1 models perform best with low temperatures (e.g., 0.1 - 0.2)
with torch.no_grad():
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

# 6. Decode and display the result
response = tokenizer.decode(output_tokens[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(response)

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/99.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/5.06M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/7.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/547 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/145 [00:00<?, ?B/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `mamba_chunk_scan_combined` is falling back to its reference PyTorch implementation because `mamba_ssm` is not installed. This is correct but much slower; install `mamba_ssm` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `selective_state_update` is falling back to its reference PyTorch implementation because `mamba_ssm` is not installed. This is correct but much slower; install `mamba_ssm` for the optimized kernel.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing 

Quantum computing is a type of computing that uses the principles of quantum mechanics to process information. Unlike traditional computers, which use bits as the smallest unit of data (represented as either 0 or 1), quantum computers use quantum bits, or qubits. Qubits can exist in a state of 0, 1, or both at the same time, thanks to a phenomenon called superposition. This allows quantum computers to process a vast number of possibilities simultaneously.

Another key concept is entanglement, where qubits can be linked together so that the state of one (0 or 1) is directly related to the state of another, no matter how far apart they are. This enables quantum computers to solve certain problems much faster than classical computers.

Quantum computing holds the potential to revolutionize fields like cryptography, drug discovery, and complex system simulations by solving problems that are currently intractable for classical computers. However, quantum computing is still in its early stag

In [ ]:
!pip install peft bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 107.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 48.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [trl]


In [ ]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 18.9 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
import os
import bitsandbytes as bnb
import pandas as pd
import torch
import torch.nn as nn
import transformers
from datasets import load_dataset
from peft import (
    LoraConfig,
    PeftConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)



In [ ]:
model_id = "tiiuae/Falcon-H1-3B-Instruct"

# 2. 4-Bit Quantization Config (Must use float16 on T4 GPUs)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16  # Force float16 compute for T4 GPU compatibility
)

# 3. Load Tokenizer & Fresh Base Model (Do NOT wrap with get_peft_model manually)
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16  # Force float16 model precision for T4 GPU compatibility
)

model = prepare_model_for_kbit_training(model)

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/99.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/5.06M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/7.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/43.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/547 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/145 [00:00<?, ?B/s]

In [ ]:
from peft import LoraConfig, TaskType

config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "in_proj", "down_proj", "up_proj", "gate_proj"]
)

In [ ]:
# model = get_peft_model(model, config)
# model.print_trainable_parameters()


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# 5. Load Sample Dataset
dataset = load_dataset("stanfordnlp/imdb", split="train[:1000]")

dataset[0]


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [ ]:
from trl import SFTConfig, SFTTrainer

# 6. Configure SFTConfig (Disable fp16/bf16 mixed-precision scaling to prevent NotImplementedError on T4 with BFloat16/Float16 conflicts)
sft_config = SFTConfig(
    output_dir="./falcon_h1_qlora_output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=100,
    fp16=False,   # Disable mixed-precision scaling to prevent conflict
    bf16=False,
    optim="paged_adamw_8bit",
    save_strategy="steps",
    save_steps=50,
    report_to="none",
    dataset_text_field="text",
    max_length=512
)

# 7. Fine-Tune with SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=config,
    processing_class=tokenizer,
    args=sft_config,
)

trainer.train()

# 8. Save the LoRA Adapter Weights
trainer.model.save_pretrained("./falcon_h1_qlora_adapter")
tokenizer.save_pretrained("./falcon_h1_qlora_adapter")

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 17}.
[transformers] `mamba_split_conv1d_scan_combined` is falling back to its reference PyTorch implementation because `mamba_ssm` is not installed. This is correct but much slower; install `mamba_ssm` for the optimized kernel.
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `mamba_chunk_scan_combined` is falling back to its reference PyTorch implementation because `mamba_ssm` is not installed. This is correct but much slower; install `mamba_ssm` for the optimized kernel.


Step,Training Loss
10,2.980606
20,2.822840
30,2.757167
40,2.736606
50,2.716728
60,2.710849
70,2.677086


Step,Training Loss
10,2.980606
20,2.822840
30,2.757167
40,2.736606
50,2.716728
60,2.710849
70,2.677086
80,2.655049
90,2.715800


In [ ]:
!pip install --upgrade torchao

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model_id = "tiiuae/Falcon-H1-3B-Instruct"
adapter_path = "./falcon_h1_qlora_adapter"
merged_save_path = "./falcon_h1_3b_merged"

# 1. Load Base Model in 16-bit precision (Float16/BFloat16)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="cpu"  # Load on CPU to save GPU VRAM during merging
)

# 2. Attach Adapter and Merge
model = PeftModel.from_pretrained(base_model, adapter_path)
merged_model = model.merge_and_unload()  # Combines LoRA weights into base weights

# 3. Save Merged Full Checkpoint
merged_model.save_pretrained(merged_save_path)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.save_pretrained(merged_save_path)

# 4. Run Inference on Merged Model
merged_model = merged_model.to("cuda").eval()

prompt = "Explain quantum computing in simple terms."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = merged_model.generate(**inputs, max_new_tokens=256, temperature=0.1)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))